Below table will provide a planned structure of this notebook:
| Section | Experiment |
|---------|-------------|
| 1 | Environment and Reproducibility |
| 2 | Williamson order-12 problem |
| 3 | QAOA Hamiltonian and Circuit |
| 4 | Classical Optimizer baseline |
| 5 | Improved training dataset |
| 6 | Shallow MLP baseline |
| 7 | Deeper MLP architecture |
| 8 | CNN architecture |
| 9 | Testing and performance comparison |
| 10 | Conclusion, Results, Plots and Future Work |

# 1. **Environment and Reproducibility**
Install the required libraries

In [3]:
# Install the required libraries
# pip install numpy scipy matplotlib pandas qiskit qiskit-aer torch scikit-learn

Import libraries

In [4]:
# Numerical computation
import numpy as np
import pandas as pd

# Scientific optimization
from scipy.optimize import minimize

# Visualization
import matplotlib.pyplot as plt

# Quantum computing
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim

# Machine learning utilities
from sklearn.model_selection import train_test_split

# Other utilities
from itertools import product
import random
import time

print("All libraries imported successfully.")

All libraries imported successfully.


**Set Random seeds and check hardware:**
`Reproducibility` is important in machine learning experiments to ensure that results can be consistently reproduced. We will compare different NN architectures.

In [5]:
# Set random seeds
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Select computation device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Computation device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.8.0
NumPy version: 2.0.2
Computation device: cpu
Running on CPU


**Our Experiment Design**: Previously, our dataset contained only `80` examples. We will aim for approximately `1000` examples in the new experiment, but first we will investigate the optimizer landscape and label quality. 

### Architecture A: Shallow MLP
$$2 \rightarrow 32 \rightarrow 16 \rightarrow 2$$

### Architecture B: Deeper MLP
$$2 \rightarrow 64 \rightarrow 64 \rightarrow 32 \rightarrow 16 \rightarrow 2$$

### Architecture C: CNN
we will first represent the two-dimensional QAOA cost landscape as a `2D (grid)` of expected costs over $\gamma$ and $\beta$ values. The CNN will learn features from that landscape and predict promising parameters. This gives the convolutional architecture a meaningful `spatial input` rather than simply applying convolutions to two scalar values.